[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/grabuffo/brain-modeling-course/blob/main/notebooks/D1_C3_NB06_active_rotator_mean_field.ipynb)

# Day 1 · Chapter 3 · Notebook 06 — Mean field of the active rotators: exact, and approximate

**Goal.** Write equations for the order parameter $Z=Re^{i\Psi}$ of the active-rotator network of NB04, as we
did for the QIF network in NB05.

There is a twist. In NB05 the neurons differed in their excitability (heterogeneity, fixed in time) and
two equations were *exact*. Here the units are identical but each receives its own **noise**, and noise
changes the story:

* the exact mean field is not 2 equations but an **infinite hierarchy** (Kuramoto–Daido), which we truncate
  at $K$ equations;
* a **2-equation reduction** (Ott–Antonsen) exists, but with noise it is an **approximation**.

We will compare network, exact hierarchy and 2D reduction, and find where the approximation fails. The
2D model is a popular building block of whole-brain models, so knowing what it leaves out matters
(remember: a model is a map, NB00).

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# On Google Colab this downloads the course code; on your laptop it does nothing.
import sys, os
if "google.colab" in sys.modules and not os.path.exists("brain-modeling-course"):
    !git clone -q https://github.com/grabuffo/brain-modeling-course.git
    %cd brain-modeling-course/notebooks
    !pip -q install numba ipywidgets
sys.path.insert(0, os.path.abspath(".."))   # so that `import src` works

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from src import plotting
from src.plotting import slider
plotting.setup()
from src import networks, meanfield
from src.plotting import show_movie
from matplotlib.animation import FuncAnimation
os.makedirs('../figures/videos', exist_ok=True)

## 1. From phases to a distribution of phases

Network (NB04): $\;\dfrac{d\varphi_j}{dt} = \omega + a\sin\varphi_j + J R\sin(\Psi-\varphi_j) + \sigma\,\xi_j(t)$.

For $N\to\infty$ we stop following each unit and follow the **density** $\rho(\varphi,t)$: the fraction of
units at phase $\varphi$. It obeys a Fokker–Planck equation (drift + diffusion):

$$
\frac{\partial\rho}{\partial t} = -\frac{\partial}{\partial\varphi}\Big[\big(\omega + a\sin\varphi + J R\sin(\Psi-\varphi)\big)\rho\Big]
+ \frac{\sigma^2}{2}\frac{\partial^2\rho}{\partial\varphi^2}
$$

A function on the circle is described by its **Fourier modes**

$$
Z_k(t) = \big\langle e^{ik\varphi}\big\rangle = \int_0^{2\pi} \rho(\varphi,t)\,e^{ik\varphi}\,d\varphi, \qquad k=1,2,\dots
$$

$Z_1=Z$ is the Kuramoto order parameter; $Z_2, Z_3,\dots$ describe finer details of the shape. The density is
rebuilt from the modes as $\rho(\varphi) = \frac{1}{2\pi}\big[1+2\,\mathrm{Re}\sum_{k\ge1}Z_k e^{-ik\varphi}\big]$.

Let us take a snapshot of the network (collective-oscillation regime of NB04) and rebuild its density with
more and more modes:

In [ ]:
rng = np.random.default_rng(3)
N = 5000
phi0 = rng.uniform(0, 2 * np.pi, N)                    # the same initial phases for network and mean field
regimes = {"quiet": (1.05, 0.3), "collective oscillations": (1.0, 0.5), "asynchronous": (1.0, 1.0)}
net = {}
for name, (a, s) in regimes.items():
    net[name] = networks.active_rotator_network(N=N, a=a, J=1.0, sigma=s, T=150, dt=0.01, rec_dt=0.1,
                                                n_rec=N, seed=3, phi0=phi0)

t, Z, ph = net["collective oscillations"]
i_snap = np.argmin(np.abs(t - 100))
x = np.mod(ph[i_snap], 2 * np.pi)
Zk = meanfield.phase_modes(x, 20)
pp = np.linspace(0, 2 * np.pi, 400)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(x, bins=80, density=True, color="C0", alpha=0.4, label="network")
for K, c in [(1, "C3"), (3, "C2"), (10, "k")]:
    ax[0].plot(pp, meanfield.phase_density(Zk[:K + 1], pp), c=c, lw=1.5, label=f"modes 1..{K}")
ax[0].axvline(np.pi / 2, c="C3", ls=":", lw=1); ax[0].text(np.pi / 2 + 0.05, ax[0].get_ylim()[1] * 0.9, "spike phase", color="C3", fontsize=8)
ax[0].set(xlabel=r"phase $\varphi$", ylabel=r"density $\rho$", title=f"snapshot at t = {t[i_snap]:.0f}"); ax[0].legend(fontsize=8)
ax[1].bar(np.arange(1, 21), np.abs(Zk[1:]), color="C0"); ax[1].set(xlabel="mode k", ylabel=r"$|Z_k|$", title="size of the Fourier modes")
plt.tight_layout()

One mode ($Z_1$ alone) gives only a rough bump; a few modes are needed to get the shape right; higher modes
become very small. Keep this in mind: the 2D model of section 5 tries to describe everything with $Z_1$.

## 2. The Kuramoto–Daido hierarchy

Multiply the Fokker–Planck equation by $e^{ik\varphi}$ and integrate (Buendía et al. 2021). Each mode obeys

$$
\frac{dZ_k}{dt} = Z_k\Big(ik\omega - \frac{k^2\sigma^2}{2}\Big) + \frac{ak}{2}\big(Z_{k+1}-Z_{k-1}\big)
+ \frac{Jk}{2}\big(Z_1 Z_{k-1} - \bar Z_1 Z_{k+1}\big), \qquad Z_0 = 1.
$$

* $ik\omega$: every mode rotates with the natural frequency.
* $-k^2\sigma^2/2$: **noise damps the modes**, and the fine details (high $k$) much faster than the coarse ones.
* $a$: the tilt couples mode $k$ to its neighbours $k\pm1$. $J$: the coupling acts through $Z_1$.

Because each mode depends on the next one, this is an **infinite chain**. Thanks to the $k^2$ damping we can
cut it at some $K$ (here $K=15$–$60$, chosen from $\sigma$) and get an **exact** description, up to truncation.

**Firing rate.** The rate is the flow of units through the spike phase $\varphi_s=\pi/2$:
$\;r = v(\varphi_s)\,\rho(\varphi_s) - \frac{\sigma^2}{2}\,\partial_\varphi\rho(\varphi_s)$, with
$v(\varphi)=\omega+a\sin\varphi+J\,\mathrm{Im}(Z_1e^{-i\varphi})$. It is computed from the modes
(`meanfield.rotator_rate`).

In [ ]:
def kd_setup(sigma):
    # number of modes and time step: more modes for weak noise, smaller steps for strong noise
    K = int(np.clip(6 / max(sigma, 1e-3), 15, 60))
    dt = min(2e-3, 1.0 / (0.5 * K * K * max(sigma, 1e-3) ** 2))
    return K, dt

kd = {}
for name, (a, s) in regimes.items():
    K, dtk = kd_setup(s)
    kd[name] = meanfield.kuramoto_daido(a=a, J=1.0, sigma=s, T=150, K=K, dt=dtk, rec_dt=0.1,
                                        Zk0=meanfield.phase_modes(phi0, K), return_modes=True)

def network_rate(t, ph, n_units, bin_size=0.5):
    st, _ = networks.phase_raster(t, ph)
    edges = np.arange(0, t[-1] + 1e-9, bin_size)
    return edges[:-1] + bin_size / 2, np.histogram(st, bins=edges)[0] / n_units / bin_size

def compare(models, title):
    fig, ax = plt.subplots(3, 4, figsize=(15, 9), gridspec_kw=dict(width_ratios=[1.6, 1.6, 1, 1]))
    pp = np.linspace(0, 2 * np.pi, 400); circle = np.exp(1j * pp)
    for row, (name, (a, s)) in enumerate(regimes.items()):
        t, Z, ph = net[name]
        tr, rr = network_rate(t, ph, N)
        i_snap = np.argmin(np.abs(t - 100))
        ax[row, 0].plot(t, np.abs(Z), "k", lw=0.7, label="network")
        ax[row, 1].plot(tr, rr, "k", lw=0.7)
        ax[row, 2].hist(np.mod(ph[i_snap], 2 * np.pi), bins=60, density=True, color="0.6", alpha=0.6)
        ax[row, 3].plot(circle.real, circle.imag, c="lightgray")
        ax[row, 3].plot(Z[t > 50].real, Z[t > 50].imag, "k", lw=0.5)
        for label, (tm, Zm), c in models(name, a, s):
            ax[row, 0].plot(tm, np.abs(Zm[:, 1]), c=c, lw=1.4, label=label)
            if label.startswith("hierarchy"):
                ax[row, 1].plot(tm, meanfield.rotator_rate(Zm, a=a, J=1.0, sigma=s), c=c, lw=1.4)
            j = np.argmin(np.abs(tm - 100))
            ax[row, 2].plot(pp, meanfield.phase_density(Zm[j], pp), c=c, lw=1.4)
            ax[row, 3].plot(Zm[tm > 50, 1].real, Zm[tm > 50, 1].imag, c=c, lw=1.4)
        ax[row, 0].set(ylabel="R", ylim=(0, 1.05), title=f"{name} (a = {a}, σ = {s}): coherence")
        ax[row, 1].set(ylabel="rate", title="firing rate")
        ax[row, 2].set(title="density at t = 100", xlim=(0, 2 * np.pi))
        ax[row, 3].set(aspect="equal", xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), title="Z in the complex plane"); ax[row, 3].axis("off")
    ax[0, 0].legend(fontsize=8, loc="lower right")
    for a_ in ax[-1, :2]: a_.set_xlabel("time")
    ax[-1, 2].set_xlabel(r"phase $\varphi$")
    fig.suptitle(title, y=1.0); plt.tight_layout()

compare(lambda name, a, s: [("hierarchy (exact)", kd[name], "C1")], "network (N = 5000) vs Kuramoto–Daido hierarchy")

The hierarchy reproduces the network in all three regimes: coherence, firing rate, shape of the phase
distribution, and the trajectory of $Z$ (a fixed point in the quiet and asynchronous regimes, a closed loop,
the **limit cycle**, in the oscillatory regime). The only difference is the finite-size noise of the network.

Watch the density move during the collective oscillation: the population gathers before the spike phase, then
"pours" through it as a population spike.

In [ ]:
name = "collective oscillations"; a, s = regimes[name]
t, Z, ph = net[name]; tm, Zm = kd[name]
frames = np.where((t >= 100) & (t < 140))[0][::2]
pp = np.linspace(0, 2 * np.pi, 400); bins = np.linspace(0, 2 * np.pi, 61)
fig, ax = plt.subplots(figsize=(7, 3.4))
ymax = 1.1 * max(meanfield.phase_density(Zm[j], pp).max() for j in frames)
_, _, bars = ax.hist(np.mod(ph[frames[0]], 2 * np.pi), bins=bins, density=True, color="C0", alpha=0.4, label="network")
line, = ax.plot(pp, meanfield.phase_density(Zm[frames[0]], pp), "C1", lw=2, label="hierarchy")
ax.axvline(np.pi / 2, c="C3", ls=":"); ax.text(np.pi / 2 + 0.05, 0.9 * ymax, "spike phase", color="C3", fontsize=8)
ax.set(xlim=(0, 2 * np.pi), ylim=(0, ymax), xlabel=r"phase $\varphi$", ylabel="density"); ax.legend(fontsize=8, loc="upper right")
ttl = ax.set_title("")

def update(k):
    j = frames[k]
    h, _ = np.histogram(np.mod(ph[j], 2 * np.pi), bins=bins, density=True)
    for b, hh in zip(bars, h): b.set_height(hh)
    line.set_ydata(meanfield.phase_density(Zm[j], pp))
    ttl.set_text(f"t = {t[j]:.1f}")
    return (*bars, line, ttl)

anim = FuncAnimation(fig, update, frames=len(frames), interval=60, blit=False)
plt.close(fig)
show_movie(anim, save_as="../figures/videos/NB06_phase_density.mp4")

## 3. How many modes do we need?

The $k^2\sigma^2/2$ damping decides it: noise smooths the distribution, so with strong noise a few modes
suffice; with weak noise the distribution is sharp and many modes are needed. If we cut too early the
truncated system can even give nonsense ($R>1$, or blow up).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for a_, s in zip(ax, [0.5, 0.2]):
    for K, c in [(2, "C3"), (5, "C2"), (10, "C0"), (40, "k")]:
        tk, Zk1 = meanfield.kuramoto_daido(a=1.0, sigma=s, T=100, K=K, dt=1e-3)
        ok = np.abs(Zk1) < 1.5
        a_.plot(tk[ok], np.abs(Zk1[ok]), c=c, lw=1.2 if K < 40 else 2, label=f"K = {K}")
    a_.axhline(1, c="gray", ls=":", lw=0.8)
    a_.set(xlabel="time", title=f"a = 1, σ = {s}", ylim=(0, 1.5))
ax[0].set_ylabel("R"); ax[0].legend(fontsize=8, ncol=2)
plt.tight_layout()

With $\sigma=0.5$, $K=10$ is already indistinguishable from $K=40$; $K=2$ is wrong. With $\sigma=0.2$, $K=2$
and $K=5$ blow up (the curves leave the plot), $K=10$ gives spurious fluctuations, and we need tens of modes.

## 4. The 2D reduction (Ott–Antonsen)

Can we close the chain with just $Z_1$? Ott and Antonsen (2008) noticed that if all modes are powers of the
first one,

$$
Z_k = Z^k \qquad (Z\equiv Z_1),
$$

the whole hierarchy collapses to a single complex equation (two real equations):

$$
\frac{dZ}{dt} = \Big(i\omega - \frac{\sigma^2}{2}\Big)Z + \frac a2\big(Z^2-1\big) + \frac J2 Z\big(1-|Z|^2\big)
$$

or, with $Z=Re^{i\Psi}$,
$\;\dot R = \frac R2\big[J(1-R^2)-\sigma^2\big] - \frac a2(1-R^2)\cos\Psi,\quad
\dot\Psi = \omega + \frac{a(1+R^2)\sin\Psi}{2R}$.

* $Z_k=Z^k$ means the density is a **wrapped Cauchy** distribution: the Lorentzian of NB05 seen on the circle
  (through $v=\tan(\varphi/2)$, NB02). This is no coincidence: the MPR equations *are* the Ott–Antonsen
  reduction of the QIF/theta network.
* The ansatz is **exact when the damping of mode $k$ grows like $k$**, which is what a Lorentzian spread of
  frequencies (or excitabilities, NB05) produces. Noise damps like $k^2$, which breaks $Z_k=Z^k$. So with noise
  the 2D model is an **approximation** (Homework 1 shows the exact case).

In [ ]:
oa = {}
for name, (a, s) in regimes.items():
    to, Zo = meanfield.ott_antonsen_ar(a=a, J=1.0, sigma=s, T=150, dt=1e-3, rec_dt=0.1, Z0=np.mean(np.exp(1j * phi0)))
    oa[name] = (to, Zo[:, None] ** np.arange(201)[None, :])     # modes from the ansatz Z_k = Z^k (many: the shape is sharp)

compare(lambda name, a, s: [("hierarchy (exact)", kd[name], "C1"), ("2D reduction", oa[name], "C0")],
        "network vs exact hierarchy vs 2D (Ott–Antonsen) reduction")

(The rate of the 2D model is not shown: the wrapped-Cauchy shape is not a solution with noise, and the rate
computed from it is unreliable.)

* **Quiet and asynchronous:** the 2D model finds the right *type* of state (a fixed point) but with a slightly
  wrong coherence, and its density is too narrow (the wrapped Cauchy has the wrong shape).
* **Collective oscillations:** the 2D model **misses them completely**. It settles on a fixed point while
  the network and the exact hierarchy oscillate.

## 5. Regime maps: exact vs 2D

How general is this? We compute, for many $(a,\sigma)$ at $J=1$, the size of the collective oscillation
(the spread of $Z$ around its mean in the stationary state) for the exact hierarchy and for the 2D model.
(About 10 s.)

In [ ]:
A = np.linspace(0.8, 1.2, 17); S = np.linspace(0.15, 1.2, 18)
osc = {"hierarchy": np.zeros((S.size, A.size)), "2D": np.zeros((S.size, A.size))}
rate_map = np.zeros((S.size, A.size))
for i, s in enumerate(S):
    K, dtk = kd_setup(s)
    for j, a in enumerate(A):
        tk, Zk = meanfield.kuramoto_daido(a=a, sigma=s, T=150, K=K, dt=dtk, Z0=0.5, return_modes=True)
        m = tk > 75
        Z1 = Zk[m, 1]
        osc["hierarchy"][i, j] = np.sqrt(np.mean(np.abs(Z1 - Z1.mean()) ** 2))
        rate_map[i, j] = meanfield.rotator_rate(Zk[m], a=a, sigma=s).mean()
        to, Zo = meanfield.ott_antonsen_ar(a=a, sigma=s, T=150, dt=2e-3, Z0=0.5)
        Zo = Zo[to > 75]
        osc["2D"][i, j] = np.sqrt(np.mean(np.abs(Zo - Zo.mean()) ** 2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
im = ax[0].pcolormesh(A, S, osc["hierarchy"], shading="auto", cmap="magma", vmin=0, vmax=0.7)
plt.colorbar(im, ax=ax[0], label="oscillation amplitude")
ax[0].contour(A, S, osc["hierarchy"], levels=[0.1], colors="w", linewidths=1.5)
ax[0].contour(A, S, osc["2D"], levels=[0.1], colors="cyan", linewidths=2, linestyles="--")
ax[0].plot([], [], "w", label="oscillations: exact"); ax[0].plot([], [], "cyan", ls="--", label="oscillations: 2D model")
ax[0].legend(fontsize=8, loc="upper right", facecolor="0.3", labelcolor="w")
ax[0].set(title="collective oscillations (exact hierarchy)")
im = ax[1].pcolormesh(A, S, rate_map, shading="auto", cmap="viridis")
plt.colorbar(im, ax=ax[1], label="firing rate")
ax[1].contour(A, S, osc["hierarchy"], levels=[0.1], colors="w", linewidths=1.5)
ax[1].text(1.12, 0.3, "quiet", color="w", ha="center"); ax[1].text(0.88, 0.45, "oscillations", color="k", ha="center")
ax[1].text(1.05, 1.05, "asynchronous", color="k", ha="center")
ax[1].set(title="firing rate (exact hierarchy)")
for a_ in ax:
    a_.axvline(1.0, c="0.7", ls=":", lw=1)
    for name, (a, s) in regimes.items():
        a_.plot(a, s, "o", mfc="none", mec="C2", ms=10, mew=2)
    a_.set(xlabel="tilt a   (units oscillate if a < 1, are excitable if a > 1)", ylabel="noise σ")
plt.tight_layout()

Green circles: the three regimes of NB04.

* Left of the dotted line ($a<1$) each unit is an oscillator; the population oscillates collectively when the
  noise is not too strong. The 2D model gets this region roughly right, but smaller.
* Right of the dotted line ($a>1$) each unit is **excitable**, it would never fire on its own without noise.
  Yet the exact hierarchy (and the network) show **collective oscillations** there: noise kicks a few units,
  the coupling recruits the others, the population fires together. The 2D model has **no** oscillations
  there at all.
* This strip of noise-induced collective oscillations of excitable units, next to the quiet and asynchronous
  regimes, is where Buendía et al. found the **hybrid synchronisation transition**, with bistability and
  scale-free avalanches. We will study it in Chapter 4 (NB08).

**Lesson.** A cheaper map is not wrong, but its borders move. Parameters tuned on the 2D model do not
transfer one-to-one to the spiking network, and some regimes may be missing from the map altogether.

### Explore: network vs exact hierarchy vs 2D model

$N=1000$ units (< 1 s per run).

* Start at $a=1$, $\sigma=0.5$ (oscillations). Increase $a$ to 1.04: do the network and the hierarchy still
  oscillate? And the 2D model?
* Find parameters where all three agree well. Where are they in the regime map?
* Set $\sigma=0$. Can the hierarchy and the 2D model still be compared with the network? (Think about the
  number of modes.)
* Change $J$: does the oscillatory region grow or shrink?

In [ ]:
@interact(a=slider(1.0, 0.7, 1.3, 0.01, "tilt a"),
          sigma=slider(0.5, 0.1, 1.3, 0.02, "noise sigma"),
          J=slider(1.0, 0.0, 3.0, 0.05, "coupling J"))
def explore_mean_fields(a, sigma, J):
    Nx = 1000
    p0 = np.random.default_rng(5).uniform(0, 2 * np.pi, Nx)
    t, Z, ph = networks.active_rotator_network(N=Nx, a=a, J=J, sigma=sigma, T=150, dt=0.01, rec_dt=0.1, n_rec=Nx, seed=5, phi0=p0)
    K, dtk = kd_setup(sigma)
    tk, Zk = meanfield.kuramoto_daido(a=a, J=J, sigma=sigma, T=150, K=K, dt=dtk, Zk0=meanfield.phase_modes(p0, K), return_modes=True)
    to, Zo = meanfield.ott_antonsen_ar(a=a, J=J, sigma=sigma, T=150, dt=1e-3, Z0=np.mean(np.exp(1j * p0)))
    tr, rr = network_rate(t, ph, Nx)
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8), gridspec_kw=dict(width_ratios=[1.5, 1.5, 1]))
    ax[0].plot(t, np.abs(Z), "k", lw=0.7, label="network N = 1000")
    ax[0].plot(tk, np.abs(Zk[:, 1]), "C1", lw=1.4, label=f"hierarchy (K = {K})")
    ax[0].plot(to, np.abs(Zo), "C0", lw=1.4, ls="--", label="2D model")
    ax[0].set(xlabel="time", ylabel="R", ylim=(0, 1.05), title=f"a = {a}, σ = {sigma}, J = {J}"); ax[0].legend(fontsize=8, loc="lower right")
    ax[1].plot(tr, rr, "k", lw=0.7); ax[1].plot(tk, meanfield.rotator_rate(Zk, a=a, J=J, sigma=sigma), "C1", lw=1.4)
    ax[1].set(xlabel="time", ylabel="firing rate", title="rate: network vs hierarchy")
    c = np.exp(1j * np.linspace(0, 2 * np.pi, 200))
    ax[2].plot(c.real, c.imag, c="lightgray")
    ax[2].plot(Z[t > 50].real, Z[t > 50].imag, "k", lw=0.5); ax[2].plot(Zk[tk > 50, 1].real, Zk[tk > 50, 1].imag, "C1", lw=1.4)
    ax[2].plot(Zo[to > 50].real, Zo[to > 50].imag, "C0", lw=1.4, ls="--")
    ax[2].set(aspect="equal", xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), title="Z (t > 50)"); ax[2].axis("off")
    plt.tight_layout(); plt.show()

## 6. Take-home

| | QIF network (NB05) | Active rotators with noise (here) |
|---|---|---|
| units differ by | excitability $\eta_j$, Lorentzian, fixed in time | independent noise $\sigma\xi_j(t)$ |
| damping of mode $k$ | $\propto k$ | $\propto k^2$ |
| exact mean field | **2 equations** (MPR) | infinite hierarchy, truncated at $K$ |
| 2 equations (Ott–Antonsen) | exact | approximate: misses noise-induced collective oscillations |

* A population of phase units is described by the distribution of its phases, and the distribution by its Fourier
  modes. The order parameter $Z$ is the first mode.
* With noise, the exact mean field is a chain of equations for the modes; truncated at a few tens of modes it
  matches the network, including the firing rate.
* The 2D reduction is cheap and analysable, but with noise it is a different map: its regime borders move, and the
  collective oscillations of excitable units disappear.
* **Next (Chapter 4):** bifurcations and criticality. What happens *at* the borders of these regimes?

### Homework
*To do at home, after the class. In class we use the **Explore** sliders; these exercises take the same
questions further and usually need a few lines of your own code.*
1. **When the 2D model is exact.** Remove the noise and give the units Lorentzian natural frequencies, half-width
   $\gamma=0.1$: `omega = 1 + 0.1 * np.tan(np.pi * (np.arange(1, N + 1) / (N + 1) - 0.5))`, then
   `networks.active_rotator_network(N=N, omega=omega, sigma=0.0, ...)`. Compare with
   `meanfield.ott_antonsen_ar(sigma=0.0, gamma=0.1, ...)` for $a=0$, $0.9$, $1.2$. Why does it work now?
2. **Kuramoto again.** For $a=0$ the 2D model predicts a stationary coherence $R=\sqrt{1-\sigma^2/J}$ (set
   $\dot R=0$). Plot it together with the exact Bessel-function result of NB04. Where do they differ most?
3. **Truncation.** For $\sigma=0.3$ and $\sigma=0.8$ ($a=1$), find the smallest $K$ for which $|Z_1(t)|$ agrees with
   $K=60$ within 1%. Relate your answer to the damping $k^2\sigma^2/2$.
4. **Two answers for one network?** At $a=1.07$, $\sigma=0.5$ run the hierarchy starting from an incoherent state
   (`Z0=0.1`) and from a coherent one (`Z0=0.95`). Do they end in the same state? Repeat with the network
   (different `phi0`). (Preview of NB08.)
5. **Rate from the modes.** Derive the firing-rate formula of section 2 from the Fokker–Planck equation (rate =
   probability current through $\varphi_s$).

### References
* Kuramoto Y (1984). *Chemical Oscillations, Waves, and Turbulence*. Springer. (Fourier hierarchy: Daido H (1996). *Physica D* 91:24–66.)
* Ott E, Antonsen TM (2008). Low dimensional behavior of large systems of globally coupled oscillators. *Chaos* 18:037113.
* Tyulkina IV, Goldobin DS, Klimenko LS, Pikovsky A (2018). Dynamics of noisy oscillator populations beyond the Ott–Antonsen ansatz. *Phys Rev Lett* 120:264101.
* Buendía V, Villegas P, Burioni R, Muñoz MA (2021). Hybrid-type synchronization transitions: where incipient oscillations, scale-free avalanches, and bistability live together. *Phys Rev Research* 3:023224.
* Montbrió E, Pazó D, Roxin A (2015). *Phys Rev X* 5:021028 (Ott–Antonsen ↔ Lorentzian ansatz).
* Rabuffo G, Bozzo P, Nguyen B, Depannemaecker D, Pompili M, Gollo L, Fukai T, Sorrentino P, Dalla Porta L (2025). Multiscale brain dynamics at the edge of criticality (preprint). Whole-brain model built on the 2D active-rotator mean field, with units tuned near $a\lesssim\omega$; see Chapter 5.